# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [68]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [69]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [70]:
openai = OpenAI()

In [71]:
# create some lists
todos = []
completed = []

In [72]:
# defing a finction to get todo reports
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: [red] {todo}[/red]\n"
    show(result)
    return result

In [73]:
get_todo_report() # test if the function works

''

In [74]:
def create_todos(description: list[str]) -> str:
  """Create new todos and add them to the list.
    loop through the description list and add each item to the todos list, and mark them as not completed in the completed list, then return the updated todo report
  Args:
      description (list[str]): A list of todo description.

  Returns:
      str: A report of all todos.
  """   
  todos.extend(description) # add the new todos to the list
  completed.extend([False] * len(description)) # mark the new todos as not completed by adding False to the completed list in the same length as the description list   
  return get_todo_report() 

In [94]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [76]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1:  Buy groceries
Todo #2:  Finish extra lab
Todo #3:  Eat banana

'Todo #1: [red] Buy groceries[/red]\nTodo #2: [red] Finish extra lab[/red]\nTodo #3: [red] Eat banana[/red]\n'

In [86]:
mark_complete(1, "I bought groceries.")

I bought groceries.

Todo #1: Interpret the problem setup and assume the distance between Boston and New York if not provided.
Todo #2:  Set up relative motion equations using departure times and speeds.
Todo #3:  Solve for meeting time and compute clock time.
Todo #4:  Present the answer clearly in Rich console markup.

'Todo #1: [green][strike]Interpret the problem setup and assume the distance between Boston and New York if not provided.[/strike][/green]\nTodo #2: [red] Set up relative motion equations using departure times and speeds.[/red]\nTodo #3: [red] Solve for meeting time and compute clock time.[/red]\nTodo #4: [red] Present the answer clearly in Rich console markup.[/red]\n'

In [87]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of description and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Description'
                }
            },
        "required": ["description"],
        "additionalProperties": False
    }
}

In [95]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [89]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [99]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [97]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [101]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves Dallas Houston at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [102]:
todos, completed = [], []
loop(messages)

Todo #1:  Interpret the problem setup and choose reasonable assumptions for missing information (e.g., 
route/distance between Boston and Houston/Dallas).
Todo #2:  Estimate the distance between Boston and the departure city (Houston, since phrasing suggests 
Dallas→Houston is a mistake) using a reasonable approximation.
Todo #3:  Set up relative motion equation with different departure times and speeds; solve for meeting time after 
2:00 pm.
Todo #4:  Convert the solution into an absolute clock time and present clearly.

Assumed the second train departs from Houston at 3:00 pm toward Boston (the phrase “Dallas Houston” appears to be a
typo). Also assumed both trains travel directly toward each other along the same line and maintain constant speeds.
Noted that the distance between the cities is not provided, so an estimate is required.

Todo #1: Interpret the problem setup and choose reasonable assumptions for missing information (e.g., 
route/distance between Boston and Houston/Dallas).
Todo #2:  Estimate the distance between Boston and the departure city (Houston, since phrasing suggests 
Dallas→Houston is a mistake) using a reasonable approximation.
Todo #3:  Set up relative motion equation with different departure times and speeds; solve for meeting time after 
2:00 pm.
Todo #4:  Convert the solution into an absolute clock time and present clearly.

Used a reasonable real-world estimate for BostonHouston separation: about 1,600 miles (typical great-circle 
distance is ~1,600 mi; actual rail route would be longer, but the problem gives no route). Took D = 1600 mi.

Todo #1: Interpret the problem setup and choose reasonable assumptions for missing information (e.g., 
route/distance between Boston and Houston/Dallas).
Todo #2: Estimate the distance between Boston and the departure city (Houston, since phrasing suggests 
Dallas→Houston is a mistake) using a reasonable approximation.
Todo #3:  Set up relative motion equation with different departure times and speeds; solve for meeting time after 
2:00 pm.
Todo #4:  Convert the solution into an absolute clock time and present clearly.

Let t = hours after 2:00 pm when they meet.

Boston train distance: 60t.
Houston train leaves at 3:00 pm, so it travels for (t−1) hours (requires t≥1); distance: 80(t−1).

They meet when distances sum to total distance D:

60t + 80(t−1) = 1600
→ 60t + 80t − 80 = 1600
→ 140t = 1680
→ t = 12 hours after 2:00 pm.

Todo #1: Interpret the problem setup and choose reasonable assumptions for missing information (e.g., 
route/distance between Boston and Houston/Dallas).
Todo #2: Estimate the distance between Boston and the departure city (Houston, since phrasing suggests 
Dallas→Houston is a mistake) using a reasonable approximation.
Todo #3: Set up relative motion equation with different departure times and speeds; solve for meeting time after 
2:00 pm.
Todo #4:  Convert the solution into an absolute clock time and present clearly.

t = 12 hours after 2:00 pm → meeting time is 2:00 am the next day.

Todo #1: Interpret the problem setup and choose reasonable assumptions for missing information (e.g., 
route/distance between Boston and Houston/Dallas).
Todo #2: Estimate the distance between Boston and the departure city (Houston, since phrasing suggests 
Dallas→Houston is a mistake) using a reasonable approximation.
Todo #3: Set up relative motion equation with different departure times and speeds; solve for meeting time after 
2:00 pm.
Todo #4: Convert the solution into an absolute clock time and present clearly.

Assumptions (since the problem omits a key value): the second train leaves from Houston at 3:00 pm toward Boston 
(the “Dallas Houston” wording appears to be a typo), and the Boston–Houston distance is about 1600 miles.

Let t = hours after 2:00 pm when they meet.

• Boston train distance: 60t  
• Houston train leaves 1 hour later, so it travels (t−1) hours: distance 80(t−1)

They meet when distances add to 1600:
60t + 80(t−1) = 1600  
140t − 80 = 1600  
140t = 1680  
t = 12

Meeting time: 12 hours after 2:00 pm = 2:00 am (next day).